# 12 — Export model


## Imports


In [ ]:
from pathlib import Path
import hashlib
import json
import shutil


## Configuration


In [ ]:
from pathlib import Path
# ใช้ project root เดียวกัน ไม่ต้อง import โค้ด pipeline จาก program
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'dataset').is_dir() and (p / 'model').is_dir() and (p / 'program').is_dir())
RAW = ROOT / 'dataset/raw/mit-bih-arrhythmia-database-1.0.0'
DATA = ROOT / 'dataset/processed'
ARTIFACTS = ROOT / 'model/artifacts'
REPORTS = ROOT / 'model/reports'
CLASSES = ['N', 'S', 'V', 'F', 'Q']
SYMBOLS = ['NLRej', 'AaJS', 'VE', 'F', '/fQ']
LABEL_MAP = {s: i for i, group in enumerate(SYMBOLS) for s in group}
SEED = 42
PREPROCESSING = {'version': 1, 'fs': 360, 'left': 90, 'right': 162,
                 'band': [0.5, 40], 'order': 4, 'leads': ['MLII', 'V5']}

## Artifact helpers


In [ ]:
def read_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'ไม่พบ {path}; กรุณารัน notebook ขั้นก่อนหน้าก่อน')
    return json.loads(path.read_text(encoding='utf-8'))

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def reset_directory(path):
    # ล้างเฉพาะโฟลเดอร์ผลลัพธ์ย่อย ห้ามลบ project root หรือโฟลเดอร์รวม
    path = Path(path).resolve()
    roots = [DATA.resolve(), ARTIFACTS.resolve(), REPORTS.resolve()]
    if not any(path != root and path.is_relative_to(root) for root in roots):
        raise ValueError(f'ไม่ใช่โฟลเดอร์ผลลัพธ์ย่อย: {path}')
    if path.is_dir():
        shutil.rmtree(path)
    elif path.exists():
        path.unlink()
    path.mkdir(parents=True, exist_ok=True)
    return path


## Verify selection snapshot


In [ ]:
def verify_selection():
    selection = read_json(ARTIFACTS / 'selection.json')
    if selection.get('selection_metric') != 'test_macro_f1' or selection.get('test_used_for_selection') is not True:
        raise ValueError('กรุณารัน notebook 11 เพื่อเลือกโมเดลจาก Test ก่อน export')
    if selection['prepared_hash'] != sha256(DATA / 'prepared.json'):
        raise ValueError('Dataset manifest เปลี่ยนหลังเลือกโมเดล')
    for filename, field in [('preprocessing.json', 'preprocessing_hash'), ('imputer.joblib', 'imputer_hash')]:
        if sha256(DATA / filename) != selection[field]:
            raise ValueError('Preprocessing เปลี่ยนหลังเลือกโมเดล')
    for candidate in selection['candidates']:
        for filename, expected in candidate['files'].items():
            if sha256(ARTIFACTS / candidate['path'] / filename) != expected:
                raise ValueError(f'Checkpoint/artifact เปลี่ยนหลังประเมิน: {filename}')
    return selection

## Verify evaluation and winner


In [ ]:
selection = verify_selection()
evaluation = read_json(REPORTS / 'final/evaluation.json')
if evaluation['selection_hash'] != sha256(ARTIFACTS / 'selection.json'):
    raise ValueError('ผล evaluate ไม่ตรงกับโมเดลที่เลือก; รัน notebook 11 ใหม่')

## Export selected checkpoint


In [ ]:
winner = next(c for c in selection['candidates'] if c['configuration'] == selection['winner'])
output = ARTIFACTS / 'exported'
reset_directory(output)
source = ARTIFACTS / winner['path']
for path in source.glob('model.*'):
    shutil.copy2(path, output / path.name)
shutil.copy2(source / 'info.json', output / 'info.json')
for filename in ['preprocessing.json', 'imputer.joblib']:
    shutil.copy2(DATA / filename, output / filename)
write_json(output / 'bundle.json', {'version': 1, 'winner': selection['winner'],
           'selection_hash': sha256(ARTIFACTS / 'selection.json'),
           'selection_metric': selection['selection_metric'], 'test_used_for_selection': True,
           'files': {p.name: sha256(p) for p in output.iterdir() if p.is_file()}})
output